In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

## Channel Selection and Exclusion

From the 58 Target=YES channels identified in EDA, **5 channels are permanently excluded** before any data is loaded:

| Channel | Reason |
|---|---|
| channel_61 | Lone Wolf — strong negative correlation with all others + sparse sampling |
| channel_62 | Sparse — only 5 anomaly events, statistically inverted behavior |
| channel_63 | Sparse — near-zero readings across 14 years |
| channel_64 | Sparse — perfect correlation (1.000) with channel_65, redundant |
| channel_65 | Sparse — perfect correlation (1.000) with channel_64, redundant |

**Result: 53 target channels retained for model input.**

In [2]:
BASE_PATH = r"C:\Users\Shivam Mohite\OneDrive\Desktop\Beacon\Data\ESA-Mission1\ESA-Mission1"
channels_path      = os.path.join(BASE_PATH, "channels.csv")
labels_path        = os.path.join(BASE_PATH, "labels.csv")
channels_folder    = os.path.join(BASE_PATH, "channels")

df_channels = pd.read_csv(channels_path)
df_labels   = pd.read_csv(labels_path, parse_dates=['StartTime', 'EndTime'])

# Channels to exclude
EXCLUDED_CHANNELS = ['channel_61', 'channel_62', 'channel_63', 'channel_64', 'channel_65']

TARGET_CHANNELS = sorted([
    ch for ch in df_channels[df_channels['Target'] == 'YES']['Channel'].tolist()
    if ch not in EXCLUDED_CHANNELS
])

print(f"Total Target=YES channels:  58")
print(f"Excluded (sparse):           {len(EXCLUDED_CHANNELS)}")
print(f"Final model input channels:  {len(TARGET_CHANNELS)}")
print(f"\nExcluded channels: {EXCLUDED_CHANNELS}")

Total Target=YES channels:  58
Excluded (sparse):           5
Final model input channels:  53

Excluded channels: ['channel_61', 'channel_62', 'channel_63', 'channel_64', 'channel_65']


## Loading All 53 Channel Pickle Files

Each channel is stored as a **Pandas DataFrame serialized as a pickle file** — one file per channel.
All 53 files are loaded into a dictionary `channel_series` where:
- **Key** = channel name (e.g. `channel_41`)
- **Value** = a Pandas Series with datetime index and normalized float values in [0, 1]

Timezone information is stripped from all indices during loading to ensure consistent timestamp
comparison when channels are aligned in the next step.

**Loading time: ~2–3 minutes (700+ million total readings across 53 channels)**

In [3]:
# Loading all 53 channels 
channel_series = {}
failed = []
for i, ch_name in enumerate(TARGET_CHANNELS):
    try:
        ch_path = os.path.join(channels_folder, f"{ch_name}/{ch_name}")
        series = pd.read_pickle(ch_path)[ch_name]

        # Strip timezone so all channels are comparable
        if series.index.tz is not None:
            series.index = series.index.tz_localize(None)

        channel_series[ch_name] = series

    except Exception as e:
        failed.append(ch_name)
        print(f"  ❌ Failed: {ch_name} — {e}")


print(f"\n{'='*50}")
print(f"Successfully loaded : {len(channel_series)} channels")
print(f"Failed              : {len(failed)} {failed if failed else ''}")


Successfully loaded : 53 channels
Failed              : 0 


## Resampling to 3-Minute Uniform Grid

**The problem:** Each channel has a different sampling frequency — channel_41 records every ~35 seconds
(19M readings) while sinusoidal channels record every ~50 seconds (14–16M readings).
They cannot be compared or combined directly because they don't share the same timestamps.

**The solution:** Resample every channel to a **uniform 3-minute grid** — the approximate median
sampling interval across all dense channels. For each 3-minute window, the **median** of all
readings within that window is used (not mean — median is robust to brief spike outliers).

**Result: `df_resampled`**
- Shape: **(2,454,720 rows × 53 columns)**
- One row every 3 minutes across 14 years
- Missing values: **343,778 (0.26%)** — communication blackouts and eclipse periods
- Data type: `float32`

> The 0.26% missing rate is exceptionally clean for real-world industrial telemetry.

In [4]:
# Resampling all 53 channels to 3-minute uniform grid
resampled = {}

for i, (ch_name, series) in enumerate(channel_series.items()):
    # Resample to 3-minute intervals using median — robust to outliers
    resampled[ch_name] = series.resample('3min').median()

    if (i + 1) % 10 == 0:
        print(f"Resampled {i+1}/53 channels...")

# Combine all channels into one unified DataFrame
df_resampled = pd.DataFrame(resampled)

df_resampled

Resampled 10/53 channels...
Resampled 20/53 channels...
Resampled 30/53 channels...
Resampled 40/53 channels...
Resampled 50/53 channels...


,channel_12,channel_13,channel_14,channel_15,channel_16,channel_17,channel_18,channel_19,channel_20,channel_21,...,channel_59,channel_60,channel_66,channel_70,channel_71,channel_72,channel_73,channel_74,channel_75,channel_76
datetime,,,,,,,,,,,,,,,,,,,,,
2000-01-01 00:00:00,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.353997,0.293778,0.370984,0.304747,...,0.890102,0.719301,0.992888,0.779727,0.813440,0.823007,0.936447,0.983601,0.909342,0.968793
2000-01-01 00:03:00,0.317175,0.365525,0.303237,0.130113,0.767688,0.349474,0.353997,0.292219,0.370984,0.304747,...,0.890102,0.718845,0.992888,0.779727,0.814124,0.823005,0.940092,0.983601,0.909339,0.969476
2000-01-01 00:06:00,0.317175,0.370984,0.297205,0.130113,0.766769,0.349474,0.354751,0.292219,0.371764,0.304747,...,0.890102,0.718845,0.993197,0.779953,0.813895,0.822323,0.936218,0.983143,0.908429,0.969019
2000-01-01 00:09:00,0.317175,0.359286,0.297205,0.130756,0.767688,0.349474,0.355505,0.292998,0.370984,0.313137,...,0.890102,0.718845,0.992888,0.778814,0.813666,0.823233,0.939634,0.983601,0.909568,0.968793
2000-01-01 00:12:00,0.317175,0.371764,0.297205,0.130113,0.766769,0.349474,0.354751,0.292219,0.370984,0.304747,...,0.890102,0.718845,0.992888,0.779498,0.813438,0.823462,0.935537,0.983601,0.908429,0.969474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-31 23:45:00,0.217355,0.269604,0.631070,0.313842,0.787915,0.265045,0.296706,0.225152,0.276623,0.642905,...,0.972004,0.820501,0.992579,0.777678,0.813438,0.823691,0.935537,0.981550,0.909339,0.968338
2013-12-31 23:48:00,0.217355,0.270384,0.631070,0.313842,0.786076,0.265798,0.294444,0.227492,0.275843,0.642905,...,0.972004,0.820501,0.992579,0.777904,0.813211,0.822097,0.938040,0.983601,0.908884,0.968111
2013-12-31 23:51:00,0.217355,0.270384,0.631070,0.313122,0.791592,0.265798,0.294444,0.228272,0.275843,0.643753,...,0.972004,0.820501,0.992579,0.779043,0.812756,0.823462,0.933257,0.982005,0.907972,0.968109


In [5]:
# Missing value analysis
missing_df = pd.DataFrame(df_resampled.isna().sum()).T  # shape (1 × 53)

top10_missing    = missing_df.T.sort_values(by=0, ascending=False).head(10)
bottom10_missing = missing_df.T.sort_values(by=0, ascending=False).tail(10)

print(f"Top 10 channels with most missing values:\n{top10_missing}\n")
print(f"Bottom 10 channels with least missing values:\n{bottom10_missing}\n")

# Correct denominator — total cells in the entire DataFrame
total_cells = df_resampled.shape[0] * df_resampled.shape[1]
total_missing = df_resampled.isna().sum().sum()

print(f"DataFrame shape:      {df_resampled.shape}")
print(f"Total cells:          {total_cells:,}")
print(f"Total missing values: {total_missing:,}")
print(f"Missing percentage:   {total_missing / total_cells * 100:.2f}%")

Top 10 channels with most missing values:
                0
channel_71  10805
channel_73  10804
channel_75  10803
channel_72  10800
channel_74  10799
channel_70  10798
channel_76  10797
channel_17   8915
channel_20   8915
channel_21   8915

Bottom 10 channels with least missing values:
              0
channel_60  446
channel_59  446
channel_58  446
channel_57  446
channel_45  358
channel_46  358
channel_42  358
channel_41  358
channel_43  358
channel_44  358

DataFrame shape:      (2454720, 53)
Total cells:          130,100,160
Total missing values: 343,778
Missing percentage:   0.26%


##  Missing Value Imputation

343,778 NaN values remain after resampling — slots where the satellite sent no data.
Standard imputation (mean/median substitution) is **inappropriate for time series** because
it ignores temporal ordering. Instead, a **two-stage time-aware strategy** is applied:

**Stage 1 — Forward Fill (limit = 10 steps = 30 minutes)**
Carry the last known sensor value forward for up to 10 consecutive missing steps.
Reflects physical reality: the most recent reading is the best estimate of current state.
→ **Filled 256,531 values (74.6% of all gaps)**

**Stage 2 — Channel Median Fill (gaps > 30 minutes)**
For extended blackouts (eclipse, safe-mode), use the channel's own 14-year median.
These are genuine long blackouts where no better estimate exists.
→ **Filled remaining 87,247 values (25.4% of all gaps)**

**Final result: `df_filled`**
- Shape: **(2,454,720 × 53)**
- **Zero null values**
- All channels aligned on a uniform 3-minute grid
- Data type: `float32`
- Ready for sliding window segmentation in Phase 3

In [6]:
# Step 1 — Forward fill up to 10 consecutive 3-min steps = 30 minutes
df_filled = df_resampled.ffill(limit=10)

# Step 2 — Backward fill for gaps at the very start of the series
df_filled = df_filled.bfill(limit=10)

# Step 3 — Check what remains unfilled
remaining_nulls = df_filled.isnull().sum().sum()
print(f"Before filling:  {df_resampled.isnull().sum().sum():,} missing values")
print(f"After filling:   {remaining_nulls:,} missing values")
print(f"Filled:          {df_resampled.isnull().sum().sum() - remaining_nulls:,} values")
print(f"Still missing:   {remaining_nulls:,} values")

# Step 4 — For any remaining nulls (long blackouts > 30 min)
# fill with channel median so the model doesn't break on NaN
if remaining_nulls > 0:
    print(f"\nFilling remaining {remaining_nulls:,} long-gap nulls with channel median...")
    df_filled = df_filled.fillna(df_filled.median())
    print(f"Final missing values: {df_filled.isnull().sum().sum()}")

print(f"\n{'='*50}")
print(f"✅ df_filled shape: {df_filled.shape}")
print(f"✅ Total nulls remaining: {df_filled.isnull().sum().sum()}")
print(f"\nData type of values:")
print(df_filled.dtypes.value_counts())

Before filling:  343,778 missing values
After filling:   87,247 missing values
Filled:          256,531 values
Still missing:   87,247 values

Filling remaining 87,247 long-gap nulls with channel median...
Final missing values: 0

✅ df_filled shape: (2454720, 53)
✅ Total nulls remaining: 0

Data type of values:
float32    53
Name: count, dtype: int64


In [7]:
WINDOW_SIZE = 100   # 100 timesteps × 3 minutes = 5 hours per window
STRIDE      = 10    # shift by 10 timesteps = 30 minutes between windows
print(f"  Window size : {WINDOW_SIZE} steps = {WINDOW_SIZE * 3 / 60:.1f} hours")
print(f"  Stride      : {STRIDE} steps = {STRIDE * 3} minutes between windows")
print(f"  Input shape : {df_filled.shape}")
print(f"  Expected windows ≈ {(len(df_filled) - WINDOW_SIZE) // STRIDE:,}\n")

# Convert DataFrame to numpy array — much faster for windowing
data_array = df_filled.values.astype(np.float32)

# Store the timestamp of the START of each window
window_start_times = []

# Pre-allocate the windows array for memory efficiency
n_windows = (len(data_array) - WINDOW_SIZE) // STRIDE
windows   = np.empty((n_windows, WINDOW_SIZE, data_array.shape[1]), dtype=np.float32)

for i in range(n_windows):
    start = i * STRIDE
    end   = start + WINDOW_SIZE
    windows[i] = data_array[start:end]
    window_start_times.append(df_filled.index[start])

window_start_times = pd.DatetimeIndex(window_start_times)

print(f"✅ Windowing complete!")
print(f"   windows shape      : {windows.shape}")
print(f"   → {windows.shape[0]:,} windows")
print(f"   → {windows.shape[1]} timesteps per window ({windows.shape[1]*3/60:.1f} hours)")
print(f"   → {windows.shape[2]} channels per timestep")
print(f"   Memory usage       : {windows.nbytes / 1e9:.2f} GB")
print(f"   First window start : {window_start_times[0]}")
print(f"   Last window start  : {window_start_times[-1]}")

  Window size : 100 steps = 5.0 hours
  Stride      : 10 steps = 30 minutes between windows
  Input shape : (2454720, 53)
  Expected windows ≈ 245,462

✅ Windowing complete!
   windows shape      : (245462, 100, 53)
   → 245,462 windows
   → 100 timesteps per window (5.0 hours)
   → 53 channels per timestep
   Memory usage       : 5.20 GB
   First window start : 2000-01-01 00:00:00
   Last window start  : 2013-12-31 18:30:00


## Labeling Windows as Normal or Anomaly

Now that we have 245,462 windows, we need to answer one question for each:
**"Does this window contain any anomaly?"**

We do this by cross-referencing each window's time range against `labels.csv`.
If ANY labeled anomaly for ANY of our 53 channels overlaps with the window's
time range — the window gets labeled **Anomaly (1)**. Otherwise it's **Normal (0)**.

In [8]:
print("Labeling windows as Normal or Anomaly...")
print(f"Total windows to label: {len(windows):,}\n")

# Fix timezone on labels so timestamps are comparable
df_labels_clean = df_labels.copy()
if df_labels_clean['StartTime'].dt.tz is not None:
    df_labels_clean['StartTime'] = df_labels_clean['StartTime'].dt.tz_localize(None)
if df_labels_clean['EndTime'].dt.tz is not None:
    df_labels_clean['EndTime']   = df_labels_clean['EndTime'].dt.tz_localize(None)

# Keep only labels for our 53 retained channels
df_labels_clean = df_labels_clean[
    df_labels_clean['Channel'].isin(TARGET_CHANNELS)
]

print(f"Anomaly label rows (53 channels only): {len(df_labels_clean):,}")
print(f"Unique anomaly events:                  {df_labels_clean['ID'].nunique()}")

# For each window, check if ANY labeled anomaly overlaps with it
# A window spans: window_start → window_start + (WINDOW_SIZE * 3 minutes)
WINDOW_DURATION = pd.Timedelta(minutes=WINDOW_SIZE * 3)

# Build a single array of (start, end) for all anomaly windows — fast lookup
anomaly_intervals = list(zip(
    df_labels_clean['StartTime'],
    df_labels_clean['EndTime']
))

# Label array — 0 = Normal, 1 = Anomaly
labels = np.zeros(len(windows), dtype=np.int8)

print("\nLabeling in progress (this may take 2-3 minutes)...")

for i, w_start in enumerate(window_start_times):
    w_end = w_start + WINDOW_DURATION
    # Check if any anomaly interval overlaps this window
    for a_start, a_end in anomaly_intervals:
        if a_start <= w_end and a_end >= w_start:
            labels[i] = 1
            break  # one overlap is enough — no need to check rest

    if (i + 1) % 50000 == 0:
        print(f"  Labeled {i+1:,} / {len(windows):,} windows...")

print(f"\n✅ Labeling complete!")
print(f"\nWindow label distribution:")
print(f"  Normal  windows (label=0): {(labels==0).sum():,}  ({100*(labels==0).mean():.2f}%)")
print(f"  Anomaly windows (label=1): {(labels==1).sum():,}  ({100*(labels==1).mean():.2f}%)")
print(f"\nClass imbalance ratio: 1 anomaly window per {(labels==0).sum()//(labels==1).sum()} normal windows")

Labeling windows as Normal or Anomaly...
Total windows to label: 245,462

Anomaly label rows (53 channels only): 3,559
Unique anomaly events:                  199

Labeling in progress (this may take 2-3 minutes)...
  Labeled 50,000 / 245,462 windows...
  Labeled 100,000 / 245,462 windows...
  Labeled 150,000 / 245,462 windows...
  Labeled 200,000 / 245,462 windows...

✅ Labeling complete!

Window label distribution:
  Normal  windows (label=0): 217,975  (88.80%)
  Anomaly windows (label=1): 27,487  (11.20%)

Class imbalance ratio: 1 anomaly window per 7 normal windows


## Train / Test Split

We split the 245,462 labeled windows into two separate piles:


In [ ]:
from sklearn.model_selection import train_test_split

print("Splitting dataset into train and test sets...")
print(f"Strategy: train on NORMAL windows only")
print(f"          test on BOTH normal and anomaly windows\n")

# Step 1 — Separate normal and anomaly windows
normal_idx  = np.where(labels == 0)[0]
anomaly_idx = np.where(labels == 1)[0]

print(f"Normal  windows available : {len(normal_idx):,}")
print(f"Anomaly windows available : {len(anomaly_idx):,}")

# Step 2 — Split normal windows into train (80%) and test (20%)
# We use shuffle=False to preserve time ordering — crucial for time series
train_idx, test_normal_idx = train_test_split(
    normal_idx,
    test_size=0.2,
    shuffle=False   # preserve temporal order
)

# Step 3 — Test set = 20% of normal + ALL anomaly windows
test_idx = np.concatenate([test_normal_idx, anomaly_idx])
test_idx = np.sort(test_idx)   # sort back into time order

# Step 4 — Extract the actual window arrays
X_train = windows[train_idx]
X_test  = windows[test_idx]

# Step 5 — Extract labels for the test set
y_test  = labels[test_idx]

print(f"\n{'='*50}")
print(f"✅ Split complete!")
print(f"\nTraining set:")
print(f"  X_train shape  : {X_train.shape}")
print(f"  Label          : 100% Normal (model never sees anomalies)")
print(f"\nTest set:")
print(f"  X_test shape   : {X_test.shape}")
print(f"  Normal windows : {(y_test==0).sum():,} ({100*(y_test==0).mean():.1f}%)")
print(f"  Anomaly windows: {(y_test==1).sum():,} ({100*(y_test==1).mean():.1f}%)")
print(f"\nMemory:")
print(f"  X_train : {X_train.nbytes/1e9:.2f} GB")
print(f"  X_test  : {X_test.nbytes/1e9:.2f} GB")

Splitting dataset into train and test sets...
Strategy: train on NORMAL windows only
          test on BOTH normal and anomaly windows

Normal  windows available : 217,975
Anomaly windows available : 27,487

✅ Split complete!

Training set:
  X_train shape  : (174380, 100, 53)
  Label          : 100% Normal (model never sees anomalies)

Test set:
  X_test shape   : (71082, 100, 53)
  Normal windows : 43,595 (61.3%)
  Anomaly windows: 27,487 (38.7%)

Memory:
  X_train : 3.70 GB
  X_test  : 1.51 GB


## Save Preprocessed Data to Disk

All preprocessed arrays are saved to `Data/processed/` so we never need to
recompute the entire pipeline again. Future notebooks simply load these files
directly and jump straight into model training.

### Files Saved

| File | Description |
|---|---|
| `X_train.npy` | Training windows — shape (174,380 × 100 × 53) — 3.70 GB |
| `X_test.npy` | Test windows — shape (71,082 × 100 × 53) — 1.51 GB |
| `y_test.npy` | Test labels — 0 = Normal, 1 = Anomaly |
| `window_start_times.npy` | Timestamp of the start of every window |
| `target_channels.json` | List of all 53 channel names used in the model |
| `preprocessing_summary.json` | Key parameters and statistics of the pipeline |

### Preprocessing Summary

| Parameter | Value |
|---|---|
| Source mission | ESA-Mission1 |
| Channels excluded | channel_61, 62, 63, 64, 65 |
| Channels in model | 53 |
| Resampling interval | 3 minutes |
| Window size | 100 timesteps = 5 hours |
| Stride | 10 timesteps = 30 minutes |
| Total windows | 245,462 |
| Training windows | 174,380 (100% Normal) |
| Test windows | 71,082 (61.3% Normal + 38.7% Anomaly) |

**Phase 2 — Preprocessing is now complete.**
Next: Phase 3 — LSTM Autoencoder model building in `003_ESA_MODEL.ipynb`

In [ ]:
import os

SAVE_DIR = r"C:\Users\Shivam Mohite\OneDrive\Desktop\Beacon\Data\processed"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving preprocessed data to disk...")
print(f"Save directory: {SAVE_DIR}\n")

# Save arrays
np.save(os.path.join(SAVE_DIR, 'X_train.npy'), X_train)
print(f"✅ X_train.npy saved  — shape {X_train.shape}  — {X_train.nbytes/1e9:.2f} GB")

np.save(os.path.join(SAVE_DIR, 'X_test.npy'),  X_test)
print(f"✅ X_test.npy  saved  — shape {X_test.shape}   — {X_test.nbytes/1e9:.2f} GB")

np.save(os.path.join(SAVE_DIR, 'y_test.npy'),  y_test)
print(f"✅ y_test.npy  saved  — shape {y_test.shape}   — {y_test.nbytes/1e6:.2f} MB")

np.save(os.path.join(SAVE_DIR, 'window_start_times.npy'),
        window_start_times.astype(str))
print(f"✅ window_start_times.npy saved — {len(window_start_times):,} timestamps")

# Save channel list so we know exactly which 53 channels are in the model
import json
with open(os.path.join(SAVE_DIR, 'target_channels.json'), 'w') as f:
    json.dump(TARGET_CHANNELS, f, indent=2)
print(f"✅ target_channels.json saved — {len(TARGET_CHANNELS)} channels")

# Save a quick summary
summary = {
    'window_size'          : int(WINDOW_SIZE),
    'stride'               : int(STRIDE),
    'n_channels'           : int(X_train.shape[2]),
    'n_train_windows'      : int(X_train.shape[0]),
    'n_test_windows'       : int(X_test.shape[0]),
    'n_test_normal'        : int((y_test==0).sum()),
    'n_test_anomaly'       : int((y_test==1).sum()),
    'train_anomaly_pct'    : 0.0,
    'test_anomaly_pct'     : round(float(100*(y_test==1).mean()), 2),
    'resampling_interval'  : '3 minutes',
    'excluded_channels'    : EXCLUDED_CHANNELS,
    'source_mission'       : 'ESA-Mission1',
}
with open(os.path.join(SAVE_DIR, 'preprocessing_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ preprocessing_summary.json saved")

print(f"\n{'='*50}")
print(f"🎉 Preprocessing complete! All files saved.")
print(f"\nFiles in {SAVE_DIR}:")
for fname in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {fname:<35} {size:.1f} MB")

Saving preprocessed data to disk...
Save directory: C:\Users\Shivam Mohite\OneDrive\Desktop\Beacon\Data\processed

✅ X_train.npy saved  — shape (174380, 100, 53)  — 3.70 GB
✅ X_test.npy  saved  — shape (71082, 100, 53)   — 1.51 GB
✅ y_test.npy  saved  — shape (71082,)   — 0.07 MB
✅ window_start_times.npy saved — 245,462 timestamps
✅ target_channels.json saved — 53 channels
✅ preprocessing_summary.json saved

🎉 Preprocessing complete! All files saved.

Files in C:\Users\Shivam Mohite\OneDrive\Desktop\Beacon\Data\processed:
  preprocessing_summary.json          0.0 MB
  target_channels.json                0.0 MB
  window_start_times.npy              5.4 MB
  X_test.npy                          1506.9 MB
  X_train.npy                         3696.9 MB
  y_test.npy                          0.1 MB
